# CycleGAN: Monet <-> Photo Style Transfer

An unpaired image-to-image translation implementation of **CycleGAN** (Zhu et al., 2017),
trained on the [monet2photo](https://www.kaggle.com/datasets/balraj98/monet2photo) dataset
to translate between Monet-style paintings and real photographs in both directions.

This notebook:
- Trains two Generators (`G_AB`, `G_BA`) and two Discriminators (`D_A`, `D_B`) with cycle-consistency, identity, and adversarial losses
- Uses a replay buffer and linear learning-rate decay, as in the original paper
- Checkpoints the model periodically so training can be resumed or used for inference later
- Logs losses to CSV and tracks a fixed set of sample images across epochs so you can visually compare progress
- Includes a standalone inference cell to run a trained model on a new image


## 0. Setup

In [ ]:
!pip -q install torch torchvision pillow kagglehub tqdm pandas

In [ ]:
import os
import random
import itertools
import gc
import csv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

import kagglehub

# Clear any stale GPU memory before starting
torch.cuda.empty_cache()
gc.collect()


In [ ]:
def set_seed(seed):
    """Make training runs reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


## 1. Configuration

All hyperparameters and paths live here. Note on `batch_size`: CycleGAN uses **InstanceNorm**
(normalizes each image independently), which is why the original paper trains with
`batch_size=1`. Larger batch sizes don't break anything technically, but they go against
the architecture's design and don't reflect how the paper's results were produced, so the
default here is 1.


In [ ]:
CONFIG = {
    # Training
    'num_epochs': 20,
    'batch_size': 1,
    'lr': 0.0002,
    'beta1': 0.5,
    'beta2': 0.999,
    'image_size': 128,
    'seed': 42,

    # Model
    'channels': 32,          # base channel width (paper uses 64; reduced here for lighter/faster training)
    'n_residual_blocks': 6,  # paper uses 9 for 256x256 images; 6 is fine for 128x128

    # Loss weights
    'lambda_cycle': 10.0,
    'lambda_identity': 5.0,

    # Replay buffer (stabilizes discriminator training, as in the paper)
    'buffer_size': 50,

    # LR decay: linearly decay to 0 over the second half of training (paper default)
    'decay_start_epoch': None,  # set to num_epochs // 2 below if left as None

    # Visualization / logging
    'num_fixed_samples': 3,     # how many images to track across epochs for the progress grid
    'checkpoint_every': 5,      # save a checkpoint every N epochs
    'checkpoint_dir': '/content/checkpoints',
    'results_dir': '/content/results',
    'loss_csv_path': '/content/results/loss_history.csv',
}

if CONFIG['decay_start_epoch'] is None:
    CONFIG['decay_start_epoch'] = CONFIG['num_epochs'] // 2

set_seed(CONFIG['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


## 2. Download dataset

In [ ]:
# Download the monet2photo dataset (unpaired Monet paintings <-> photographs)
dataset_path = kagglehub.dataset_download("balraj98/monet2photo")
print("Path to dataset files:", dataset_path)

# NOTE on domain naming: in this dataset, `trainA` = Monet paintings and `trainB` = photos.
# This notebook defines "A" = photos and "B" = Monet paintings (the reverse of the folder
# names) simply because that's how the original script was set up. Flip these two lines if
# you'd rather have A = Monet and B = photos; it doesn't affect correctness, only labeling.
CONFIG['dataroot_A'] = os.path.join(dataset_path, 'trainB')  # photos
CONFIG['dataroot_B'] = os.path.join(dataset_path, 'trainA')  # Monet paintings


## 3. Generator

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super(ResidualBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.InstanceNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.InstanceNorm2d(channels)
        )

    def forward(self, x):
        return x + self.block(x)


class Generator(nn.Module):
    def __init__(self, channels=32, n_residual_blocks=6):
        super(Generator, self).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(3, channels, kernel_size=7, stride=1, padding=3),
            nn.InstanceNorm2d(channels),
            nn.ReLU(inplace=True)
        )

        self.down_blocks = nn.Sequential(
            self._make_layer(channels, channels * 2),
            self._make_layer(channels * 2, channels * 4)
        )

        self.res_blocks = nn.Sequential(
            *[ResidualBlock(channels * 4) for _ in range(n_residual_blocks)]
        )

        self.up_blocks = nn.Sequential(
            self._make_layer(channels * 4, channels * 2, upsample=True),
            self._make_layer(channels * 2, channels, upsample=True)
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(channels, 3, kernel_size=7, stride=1, padding=3),
            nn.Tanh()
        )

    def _make_layer(self, in_channels, out_channels, upsample=False):
        if upsample:
            return nn.Sequential(
                nn.ConvTranspose2d(in_channels, out_channels, 3, stride=2, padding=1, output_padding=1),
                nn.InstanceNorm2d(out_channels),
                nn.ReLU(inplace=True)
            )
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride=2, padding=1),
            nn.InstanceNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.down_blocks(x)
        x = self.res_blocks(x)
        x = self.up_blocks(x)
        x = self.conv2(x)
        return x


## 4. Discriminator

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, channels=32):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, channels, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(channels, channels * 2, 4, stride=2, padding=1),
            nn.InstanceNorm2d(channels * 2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(channels * 2, channels * 4, 4, stride=2, padding=1),
            nn.InstanceNorm2d(channels * 4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(channels * 4, channels * 8, 4, padding=1),
            nn.InstanceNorm2d(channels * 8),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(channels * 8, 1, 4, padding=1)
        )

    def forward(self, x):
        return self.model(x)


## 5. Replay Buffer

As in the original CycleGAN paper, discriminators are updated using a history of previously
generated fake images (not just the current batch). This reduces oscillation during training.


In [ ]:
class ReplayBuffer:
    def __init__(self, max_size=50):
        self.max_size = max_size
        self.data = []

    def push_and_pop(self, batch):
        """Given a batch of freshly generated (detached) fake images, returns a batch that's
        a 50/50 mix of the fresh images and previously seen ones, updating the buffer in place."""
        to_return = []
        for element in batch:
            element = element.unsqueeze(0)
            if len(self.data) < self.max_size:
                self.data.append(element)
                to_return.append(element)
            else:
                if np.random.uniform(0, 1) > 0.5:
                    i = np.random.randint(0, self.max_size)
                    to_return.append(self.data[i].clone())
                    self.data[i] = element
                else:
                    to_return.append(element)
        return torch.cat(to_return)


## 6. Dataset

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, root_A, root_B, transform=None):
        self.root_A = root_A
        self.root_B = root_B
        self.transform = transform

        self.files_A = sorted(os.listdir(root_A))
        self.files_B = sorted(os.listdir(root_B))

    def __getitem__(self, index):
        img_A = Image.open(os.path.join(self.root_A, self.files_A[index % len(self.files_A)])).convert('RGB')
        img_B = Image.open(os.path.join(self.root_B, self.files_B[index % len(self.files_B)])).convert('RGB')

        if self.transform:
            img_A = self.transform(img_A)
            img_B = self.transform(img_B)

        return {'A': img_A, 'B': img_B}

    def __len__(self):
        return max(len(self.files_A), len(self.files_B))


## 7. Visualization & Logging Helpers

`save_sample_grid` saves the fixed-sample comparison grid to disk every epoch (as well as
displaying it), so the progression is preserved even if the notebook output is cleared,
and can be turned into a GIF later.


In [ ]:
def denormalize(tensor):
    return (tensor.cpu().detach().permute(1, 2, 0) * 0.5 + 0.5).clamp(0, 1)


def save_sample_grid(real_A, fake_B, real_B, fake_A, epoch, results_dir, num_images=3):
    os.makedirs(results_dir, exist_ok=True)
    num_images = min(num_images, len(real_A))
    num_rows = 4

    plt.figure(figsize=(12, num_rows * 3))
    rows = [
        (real_A, 'Real A'),
        (fake_B, 'Fake B'),
        (real_B, 'Real B'),
        (fake_A, 'Fake A'),
    ]
    for row_idx, (images, label) in enumerate(rows):
        for i in range(num_images):
            plt.subplot(num_rows, num_images, row_idx * num_images + i + 1)
            plt.imshow(denormalize(images[i]))
            plt.title(label)
            plt.axis('off')

    plt.suptitle(f'Epoch {epoch + 1}')
    plt.tight_layout()
    save_path = os.path.join(results_dir, f'epoch_{epoch + 1:03d}.png')
    plt.savefig(save_path, dpi=100)
    plt.show()
    plt.close()


def save_checkpoint(epoch, G_AB, G_BA, D_A, D_B, optimizer_G, optimizer_D_A, optimizer_D_B, checkpoint_dir):
    os.makedirs(checkpoint_dir, exist_ok=True)
    path = os.path.join(checkpoint_dir, f'checkpoint_epoch_{epoch + 1}.pth')
    torch.save({
        'epoch': epoch + 1,
        'G_AB': G_AB.state_dict(),
        'G_BA': G_BA.state_dict(),
        'D_A': D_A.state_dict(),
        'D_B': D_B.state_dict(),
        'optimizer_G': optimizer_G.state_dict(),
        'optimizer_D_A': optimizer_D_A.state_dict(),
        'optimizer_D_B': optimizer_D_B.state_dict(),
        'config': CONFIG,
    }, path)
    print(f"Saved checkpoint: {path}")


def append_loss_history(history, csv_path):
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    pd.DataFrame(history).to_csv(csv_path, index=False)


## 8. Training Loop

Key differences from a minimal CycleGAN implementation:
- Discriminators are updated using the **replay buffer**, not just the freshest fake images
- Learning rate **linearly decays to 0** over the second half of training
- A **fixed set of sample images** is used for the visual grid every epoch (so you're
  comparing the same images over time, not a random batch)
- Per-epoch losses (broken into GAN / cycle / identity components) are logged to CSV
- Checkpoints are saved periodically


In [ ]:
def train_cyclegan(config):
    print("=== Starting CycleGAN Training ===")
    print(f"Using device: {device}")

    G_AB = Generator(channels=config['channels'], n_residual_blocks=config['n_residual_blocks']).to(device)
    G_BA = Generator(channels=config['channels'], n_residual_blocks=config['n_residual_blocks']).to(device)
    D_A = Discriminator(channels=config['channels']).to(device)
    D_B = Discriminator(channels=config['channels']).to(device)

    G_AB.train()
    G_BA.train()

    criterion_GAN = nn.MSELoss()
    criterion_cycle = nn.L1Loss()
    criterion_identity = nn.L1Loss()

    optimizer_G = optim.Adam(
        itertools.chain(G_AB.parameters(), G_BA.parameters()),
        lr=config['lr'], betas=(config['beta1'], config['beta2'])
    )
    optimizer_D_A = optim.Adam(D_A.parameters(), lr=config['lr'], betas=(config['beta1'], config['beta2']))
    optimizer_D_B = optim.Adam(D_B.parameters(), lr=config['lr'], betas=(config['beta1'], config['beta2']))

    def lr_lambda(epoch):
        decay_start = config['decay_start_epoch']
        num_epochs = config['num_epochs']
        if epoch < decay_start:
            return 1.0
        return 1.0 - (epoch - decay_start) / float(num_epochs - decay_start + 1)

    scheduler_G = optim.lr_scheduler.LambdaLR(optimizer_G, lr_lambda=lr_lambda)
    scheduler_D_A = optim.lr_scheduler.LambdaLR(optimizer_D_A, lr_lambda=lr_lambda)
    scheduler_D_B = optim.lr_scheduler.LambdaLR(optimizer_D_B, lr_lambda=lr_lambda)

    transform = transforms.Compose([
        transforms.Resize(config['image_size']),
        transforms.CenterCrop(config['image_size']),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    dataset = ImageDataset(config['dataroot_A'], config['dataroot_B'], transform)
    dataloader = DataLoader(dataset, batch_size=config['batch_size'], shuffle=True)

    # Fixed samples for consistent progress visualization across epochs
    n_fixed = min(config['num_fixed_samples'], len(dataset))
    fixed_items = [dataset[i] for i in range(n_fixed)]
    fixed_real_A = torch.stack([item['A'] for item in fixed_items]).to(device)
    fixed_real_B = torch.stack([item['B'] for item in fixed_items]).to(device)

    fake_A_buffer = ReplayBuffer(config['buffer_size'])
    fake_B_buffer = ReplayBuffer(config['buffer_size'])

    history = {
        'epoch': [], 'loss_G': [], 'loss_D': [],
        'loss_GAN': [], 'loss_cycle': [], 'loss_identity': []
    }

    for epoch in range(config['num_epochs']):
        epoch_losses = {'loss_G': 0.0, 'loss_D': 0.0, 'loss_GAN': 0.0, 'loss_cycle': 0.0, 'loss_identity': 0.0}
        n_batches = len(dataloader)

        pbar = tqdm(enumerate(dataloader), total=n_batches, desc=f'Epoch {epoch + 1}/{config["num_epochs"]}')
        for i, batch in pbar:
            real_A = batch['A'].to(device)
            real_B = batch['B'].to(device)

            # ---------------------- Train Generators ----------------------
            optimizer_G.zero_grad()

            fake_B = G_AB(real_A)
            fake_A = G_BA(real_B)

            loss_id_A = criterion_identity(G_BA(real_A), real_A)
            loss_id_B = criterion_identity(G_AB(real_B), real_B)
            loss_identity = loss_id_A + loss_id_B

            pred_fake_B = D_B(fake_B)
            pred_fake_A = D_A(fake_A)
            loss_GAN_AB = criterion_GAN(pred_fake_B, torch.ones_like(pred_fake_B))
            loss_GAN_BA = criterion_GAN(pred_fake_A, torch.ones_like(pred_fake_A))
            loss_GAN = loss_GAN_AB + loss_GAN_BA

            recovered_A = G_BA(fake_B)
            recovered_B = G_AB(fake_A)
            loss_cycle_A = criterion_cycle(recovered_A, real_A)
            loss_cycle_B = criterion_cycle(recovered_B, real_B)
            loss_cycle = loss_cycle_A + loss_cycle_B

            loss_G = loss_GAN + loss_cycle * config['lambda_cycle'] + loss_identity * config['lambda_identity']
            loss_G.backward()
            optimizer_G.step()

            # -------------------- Train Discriminators ---------------------
            optimizer_D_A.zero_grad()
            optimizer_D_B.zero_grad()

            fake_A_ = fake_A_buffer.push_and_pop(fake_A.detach())
            fake_B_ = fake_B_buffer.push_and_pop(fake_B.detach())

            pred_real_A = D_A(real_A)
            pred_fake_A_ = D_A(fake_A_)
            loss_D_A = criterion_GAN(pred_real_A, torch.ones_like(pred_real_A)) + \
                       criterion_GAN(pred_fake_A_, torch.zeros_like(pred_fake_A_))

            pred_real_B = D_B(real_B)
            pred_fake_B_ = D_B(fake_B_)
            loss_D_B = criterion_GAN(pred_real_B, torch.ones_like(pred_real_B)) + \
                       criterion_GAN(pred_fake_B_, torch.zeros_like(pred_fake_B_))

            loss_D = (loss_D_A + loss_D_B) * 0.5
            loss_D.backward()

            optimizer_D_A.step()
            optimizer_D_B.step()

            epoch_losses['loss_G'] += loss_G.item()
            epoch_losses['loss_D'] += loss_D.item()
            epoch_losses['loss_GAN'] += loss_GAN.item()
            epoch_losses['loss_cycle'] += loss_cycle.item()
            epoch_losses['loss_identity'] += loss_identity.item()

            pbar.set_postfix({'G_loss': f'{loss_G.item():.3f}', 'D_loss': f'{loss_D.item():.3f}'})

        # End-of-epoch LR decay step
        scheduler_G.step()
        scheduler_D_A.step()
        scheduler_D_B.step()

        # Log average losses for the epoch
        history['epoch'].append(epoch + 1)
        for key in ['loss_G', 'loss_D', 'loss_GAN', 'loss_cycle', 'loss_identity']:
            history[key].append(epoch_losses[key] / n_batches)
        append_loss_history(history, config['loss_csv_path'])

        # Visualize fixed samples
        G_AB.eval()
        G_BA.eval()
        with torch.no_grad():
            fixed_fake_B = G_AB(fixed_real_A)
            fixed_fake_A = G_BA(fixed_real_B)
        save_sample_grid(fixed_real_A, fixed_fake_B, fixed_real_B, fixed_fake_A, epoch, config['results_dir'])
        G_AB.train()
        G_BA.train()

        # Checkpoint periodically and on the final epoch
        if (epoch + 1) % config['checkpoint_every'] == 0 or (epoch + 1) == config['num_epochs']:
            save_checkpoint(epoch, G_AB, G_BA, D_A, D_B, optimizer_G, optimizer_D_A, optimizer_D_B,
                             config['checkpoint_dir'])

    print("Training completed!")
    return history, G_AB, G_BA, D_A, D_B


## 9. Run Training

In [ ]:
torch.cuda.empty_cache()
gc.collect()

history, G_AB, G_BA, D_A, D_B = train_cyclegan(CONFIG)


## 10. Plot Loss Curves

In [ ]:
loss_df = pd.read_csv(CONFIG['loss_csv_path'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(loss_df['epoch'], loss_df['loss_G'], label='Generator loss')
axes[0].plot(loss_df['epoch'], loss_df['loss_D'], label='Discriminator loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Generator vs Discriminator Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(loss_df['epoch'], loss_df['loss_GAN'], label='GAN loss')
axes[1].plot(loss_df['epoch'], loss_df['loss_cycle'], label='Cycle loss')
axes[1].plot(loss_df['epoch'], loss_df['loss_identity'], label='Identity loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Generator Loss Components')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['results_dir'], 'loss_curves.png'), dpi=100)
plt.show()


## 11. Inference on a New Image

Loads a saved checkpoint and runs one of the trained generators on a new image.
Set `direction='AB'` to translate photo -> Monet-style, or `'BA'` for Monet -> photo
(based on the domain naming set up in Section 2).


In [ ]:
def load_generator(checkpoint_path, direction='AB', config=CONFIG, map_location=None):
    map_location = map_location or device
    checkpoint = torch.load(checkpoint_path, map_location=map_location)
    ckpt_config = checkpoint.get('config', config)

    generator = Generator(
        channels=ckpt_config['channels'],
        n_residual_blocks=ckpt_config['n_residual_blocks']
    ).to(map_location)

    state_key = 'G_AB' if direction == 'AB' else 'G_BA'
    generator.load_state_dict(checkpoint[state_key])
    generator.eval()
    return generator


def run_inference(checkpoint_path, image_path, direction='AB', image_size=None, output_path=None):
    image_size = image_size or CONFIG['image_size']
    generator = load_generator(checkpoint_path, direction=direction)

    transform = transforms.Compose([
        transforms.Resize(image_size),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    input_image = Image.open(image_path).convert('RGB')
    input_tensor = transform(input_image).unsqueeze(0).to(device)

    with torch.no_grad():
        output_tensor = generator(input_tensor)

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(denormalize(input_tensor[0]))
    axes[0].set_title('Input')
    axes[0].axis('off')
    axes[1].imshow(denormalize(output_tensor[0]))
    axes[1].set_title(f'Output ({direction})')
    axes[1].axis('off')
    plt.tight_layout()

    if output_path:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        plt.savefig(output_path, dpi=100)
    plt.show()

    return output_tensor


In [ ]:
# Example usage (uncomment and adjust paths once you have a trained checkpoint
# and a sample image to test on):

checkpoint_path = os.path.join(CONFIG['checkpoint_dir'], f"checkpoint_epoch_{CONFIG['num_epochs']}.pth")
sample_image_path = os.path.join(CONFIG['dataroot_A'], sorted(os.listdir(CONFIG['dataroot_A']))[0])
_ = run_inference(checkpoint_path, sample_image_path, direction='AB',
                   output_path=os.path.join(CONFIG['results_dir'], 'inference_example.png'))
